In [ ]:
import os
import rasterio
from importlib_resources import files
from pathlib import Path
from rasterio.enums import Resampling
from tqdm import tqdm

from beak.utilities.raster_processing import unify_raster_grids
from beak.utilities.io import save_raster, create_file_list, check_path, create_file_folder_list


**User inputs**
1. Select the root folder where the rasters are stored.
2. Then, go down to select the subfolders that contain the rasters to be unified.<p>

The script will search for all rasters and store them in relative paths.

In [ ]:
BASE_PATH = files("beak.data")
BASE_SPATIAL = "102003_500"
BASE_EXTENT = "upper_midwest_cobalt_nickel"
BASE_RASTER = BASE_PATH / "BASE_RASTERS" / "EPSG_102003_RES_500_UpperMidwest_ENV_TA2_HM6_NAT.tif"
BASE_EVIDENCE = "geophysics"

# Export
# Some data sets are already **LOG** scaled and need special actions. They will be unified and stored in a separate folder.
PATH_INPUT = BASE_PATH / "RAW" / BASE_EVIDENCE / "national_scale"

PATH_ROOT = BASE_PATH / "PROCESSED" / str("regional" + "_" + BASE_SPATIAL + "_" + BASE_EXTENT)
PATH_EXPORT = PATH_ROOT / "env_area" / "unified_ta2_hm6_nat" / BASE_EVIDENCE
PATH_EXPORT_LOG = PATH_ROOT / "env_area" / "unified_ta2_hm6_nat_scaled_log" / BASE_EVIDENCE

CURRENT_DIR = Path(os.getcwd()) / PATH_EXPORT.name
OUT_FOLDER = PATH_EXPORT
OUT_FOLDER_LOG = PATH_EXPORT_LOG

print(f"Input folder: {PATH_INPUT}")
print(f"Export folder: {OUT_FOLDER}")

Select subfolders to be scaled.

In [ ]:
for folder in os.listdir(PATH_INPUT):
  if os.path.isdir(os.path.join(PATH_INPUT, folder)):
    print(folder)


In [ ]:
SELECTION = ["gravity", 
             "magnetic",
             "magnetotelluric",
             "radiometric",
             "seismic"]

input_folders = [PATH_INPUT / folder for folder in SELECTION]

print("Selected folders:")
for folder in input_folders:
  print(folder)

**Select files**

In [ ]:
# Create file list
file_list = []
for folder in input_folders:
  files = create_file_list(folder, recursive=True)
  file_list.extend(files)

# Remove CONUS 2021 data
file_list = [file for file in file_list if "CONUS_2021" not in str(file)]

# Create file list for log scaled data
log_files = [file.stem for file in file_list if "Log" in file.stem]

# Show results
print(f"Found {len(file_list)} files including {len(log_files)} log scaled files.")


**Run unification**

In [ ]:
# Select to write output file
dry_run = False

if dry_run:
  print("Dry run, no files will be written.\n")

# Unify
for file in tqdm(file_list, total=len(file_list)):
    folder_relative = os.path.relpath(file.parent, PATH_INPUT)

    raster = rasterio.open(file)
    unified_raster, unified_meta = unify_raster_grids(BASE_RASTER, [file], resampling_method=Resampling.bilinear, same_extent=True, same_shape=True)[0]

    if not dry_run:
      if file.stem in log_files:
        log_folder = OUT_FOLDER_LOG / str(folder_relative)
        log_out_path = log_folder / file.name
        check_path(Path(os.path.dirname(log_out_path)))
        save_raster(log_out_path, array=unified_raster, dtype="float32", metadata=unified_meta)
      else:
        output_folder = OUT_FOLDER / str(folder_relative)
        out_path = output_folder / file.name
        check_path(Path(os.path.dirname(out_path)))
        save_raster(out_path, array=unified_raster, dtype="float32", metadata=unified_meta)
